<img src="https://github.com/moroneyt/MXB301/raw/main/resources/qutlogo.jpg">

# MXB301 Mathematics of AI
# Lesson 10: Random Walks and Diffusion Processes

#### Tim Moroney, 2026

A lesson all about diffusion.

# Introduction

This week we explore the mathematics of diffusion processes. Diffusion is a central topic in so many areas of maths and science, so it's a great topic to delve into. But particularly relevant for this unit is the application of diffusion to so-called _denoising diffusion_ models for image generation. You will have heard of the likes of [Stable Diffusion](https://stablediffusionweb.com/) that use this process to almost magically transform _pure randomness_ into photo-realistic images by running a diffusion process _in reverse_.  Over the next few lessons we will build up to a working generative model.



# Package management

In [ ]:
import Pkg
if haskey(ENV, "COLAB_GPU") # check if we're on Colab
  if !isfile("/content/MXB301_2026_01_CPU.tgz") # check if we've already downloaded
    # download precompiled Julia environment for Colab
    run(`gdown https://drive.google.com/uc\?id=1mT9XFadzdfK8CWb5a7BYLUkd2RTi2eZc`)

    # replace Colab's Julia environment with downloaded version
    run(`rm -rf /root/.julia`)
    run(`tar -xzf MXB301_2026_01_CPU.tgz -C /root`)
  end
else
  # For any other machine we install the packages in the usual way
  Pkg.activate(".")
  Pkg.add(["CairoMakie", "CodecZlib", "ColorSchemes", "ComponentArrays", "CondaPkg",
           "DifferentiationInterface", "Distributions", "Downloads", "FiniteDiff", "ForwardDiff",
           "HTTP", "JLD2", "LaTeXStrings", "LinearAlgebra", "Lux", "MKL", "MLUtils", "NNlib",
           "NLSolversBase", "OneHotArrays", "Optim", "PythonCall", "QuadGK", "Random",
           "SpecialFunctions", "Statistics", "StatsBase", "ToeplitzMatrices", "Zygote"])
end

using CairoMakie
using DifferentiationInterface
using LaTeXStrings
using LinearAlgebra
using Lux
using Random
using SpecialFunctions
using Statistics

using ComponentArrays: ComponentVector
using Distributions: Normal, Exponential
using Downloads: download
using ForwardDiff: Dual, partials
using JLD2: jldopen
using MLUtils: DataLoader, rand_like, randn_like
using NLSolversBase: only_fg
using NNlib: softmax, sigmoid, scatter as scattergrad, conv, ∇conv_filter, ∇conv_data, DenseConvDims
using OneHotArrays: onehot, onehotbatch, onecold
using StatsBase: crossentropy, sample, Weights
using ToeplitzMatrices: Toeplitz, Hankel
using QuadGK: quadgk

import CodecZlib
import ColorSchemes
import FiniteDiff
import HTTP
import MKL
import Optim
import Zygote

# Set the random seed for reproducibility
rng = Random.seed!(0)

# SVG format scales properly in web pages and PDFs
CairoMakie.activate!(type = "svg")

# Problem setup

To motivate our presentation we consider a highly idealised problem.  Suppose we have built a convolutional variational autoencoder (CVAE), as we learned about in the last lesson.  To keep the focus on the new mathematics, we will strip the problem down to its bare essence.  Later on we can add some of the complexity back.

Assumptions:

*  The only kinds of images our CVAE is trained on are cat images.
* The distribution of all possible cat images includes only three kinds of cats: sad cats, angry cats and happy cats.
*  The latent representation of cat images is one-dimensional.  That is, our CVAE encodes a cat image to a single value on the real number line.

# Ground truth distribution
Pictured below is the ground truth distribution of cat latents we are dreaming up for our simplified scenario. You can just admire the figure, no need to dwell on the code to generate it.

In [ ]:
function happypdf(z, t; θ, σ, λ)
    # exponential distribution initially
    a = exp(-θ*t)
    σₜ = sqrt((σ^2/(2θ))*(1 - a^2))
    λₜ = a*λ
    ξ = z/σₜ - σₜ/λₜ
    Φ = 0.5*erfc(-ξ / sqrt(2))
    return 1/λₜ * exp(σₜ^2/(2λₜ^2) - z/λₜ) * Φ
end

function unhappypdf(z, t; θ, σ, z0, s)
    # normal distribution initially
    μₜ = z0 * exp(-θ*t)
    sₜ² = s^2 * exp(-2θ*t) + σ^2/(2θ) * (1 - exp(-2θ*t))
    return exp(-0.5 * (z - μₜ)^2 / sₜ²) / sqrt(2π * sₜ²)
end

catpdf(z,t) = 0.5*happypdf(z, t; θ=1/2, σ=1, λ=0.8) +
              0.2*unhappypdf(z, t; θ=1/2, σ=1, z0=-1.2, s=0.1) +
              0.3*unhappypdf(z, t; θ=1/2, σ=1, z0=-0.8, s=0.1)

lines(-3..3, z->catpdf(z, 0), color=:black, label=L"p_{cat}",
  axis=(title = "Ground truth latent cat distribution",
  xlabel=L"z", xlabelsize=20, ylabel=L"p(z)", ylabelsize=20,
  xticks=([-3,-1.2,-0.8,0.8,3], ["\n-3","😿\n-1.2","😾\n-0.8","😺\n0.8","\n3"]), xticklabelsize=16)
)

# Distribution properties

Since we are imagining this distribution was sculpted by a VAE, with its KL divergence term in the loss function, it should be broadly consistent with $\mathcal{N}(0,I)$.

[A word on terminology: of course real latent image spaces are not one dimensional, so to keep our notation general we will refer to the normal distribution as $\mathcal{N}(0,I)$ throughout.  For our particular made-up cat example, the latent dimension is only one, so we are actually dealing with $\mathcal{N}(0,1)$.]

Indeed we can calculate the mean and variance of this distribution using numerical integration to confirm they come out as roughly zero and one respectively.


In [ ]:
catmean = first(quadgk(z->z*catpdf(z, 0), -5, 10))  # should be roughly zero
catvar = first(quadgk(z->(z-catmean)^2*catpdf(z, 0), -5, 10))  # should be roughly one
catmean, catvar

# Compared to $\mathcal{N}(0,I)$
Overlaying the standard normal distribution confirms that any random draw from the cat distribution would look consistent with a random draw from $\mathcal{N}(0,I)$.  But obviously there is more structure in the cat distribution than $\mathcal{N}(0,I)$.  The reconstruction term in the VAE loss function ensures that whatever way our cat distribution differs from $\mathcal{N}(0,I)$ is precisely so that it can reconstruct cat pictures from latents.

As far as the structure in this example is concerned, we observe that negative cat emotions correspond to negative latent values, and positive cat emotions correspond to positive. The distribution shows some overlap with sad cats and angry cats (sad, angry cats!) but there is no such thing as a sad, happy cat (or an angry, happy cat).  In particular the region of latent space between about $-0.5$ and $0$ has essentially zero probability.  So there is semantic structure encoded in the distribution, as we expect of our latent space.

Again we emphasise that this distribution is completely made-up as an example only, but we might as well choose an interesting-looking distribution like this one to experiment with. In truth, it's just a combination of two Gaussians and an exponential, so we have the analytic formulas available when we need them.

In [ ]:
lines!(-3..3, z->catpdf(z, 5), label=L"N(0,1)")
axislegend()
current_figure()

# Sampling from the latent cat distribution

The big idea underpinning diffusion-based generative image models is that the latent distribution $p(z)$ can be transformed into a standard normal $\mathcal{N}(0,I)$ distribution _and vice versa_.  That is, there exists a two-way mapping from the ground truth to $\mathcal{N}(0,I)$.  The cat distribution we've invented is just simple enough that we can solve for this mapping analytically (next lesson's exercises!).  Here it is visualised as an evolution over a time variable $t$.



In [ ]:
fig = Figure(size=(800,800), fontsize=20)
tvec = [0 0.01 0.05 ; 0.1 0.3 5]
for (pos, t) in pairs(tvec)
  lines(fig[Tuple(pos)...], -3..3, z->catpdf(z, t), axis = (xlabel=L"z", ylabel=L"p(z)", title = "t = $t",))
end
fig

##
And as an animation in both directions.

<img width=400 src="https://github.com/moroneyt/MXB301/raw/main/resources/distribution_warp.gif">

##

Why do we want such a mapping?  It's so that we can _sample_ from this latent distribution $p(z)$.  It's easy to sample from $\mathcal{N}(0,I)$ after all.  So if we have a mapping from $\mathcal{N}(0,I)$ to the cat latent distribution, we can sample from the latter by sampling from the former, and applying the mapping to it.  Here's what the full process to generate new images looks like:

1. Sample random value from $\mathcal{N}(0,I)$
2. Apply inverse mapping to obtain sample from latent distribution
3. Apply VAE decoder to sampled latent
4. Admire generated cat picture

This is one of the main reasons that modern generative image models like Stable Diffusion are based on latent spaces rather than pure image spaces.  The VAE encoder already encourages latents to be broadly compatible with $\mathcal{N}(0,I)$.  So we might hope that it's not too difficult to learn a transformation between the latent distribution $p(z)$ and $\mathcal{N}(0,I)$.

##
If right now you're thinking "but surely mapping one distribution to another is a different matter than mapping a _sample_ from a distribution", then well done.  Indeed, the first part of our investigation into this whole story will be to clarify how these two ideas _can_ be related.

But notice how kind-of unremarkable the idea of generating new cat images now appears through this lens.  Sure, we can map one distribution to another, no big deal.  And that should give us a way to sample a value from $\mathcal{N}(0,I)$ and map it to a cat latent, and hence decode it into a never-before-seen cat picture.  Hey presto, a generative cat image model.

Keeping ourselves grounded in one dimension for this lesson, let's begin the journey.  For the rest of the lesson we will build up some necessary mathematics, without further reference to any AI application.

# Random walks

Here's a classic random walk code like you would be familiar with from MXB161.  The particle position is denoted as $z_n$, because it represents a point in latent space.  The code updates the positions over some number of steps by repeatedly applying the formula
$$
z_{n+1} = \begin{cases}
z_n - \Delta,\quad \textrm{with probability } q\\
z_n + \Delta,\quad \textrm{with probability } 1-q\,.\\
\end{cases}
$$
That is, the step direction $+$ or $-$ is chosen randomly each step with not-necessarily-equal probability.

In [ ]:
function random_walk(; z0, nsteps, Δ = 1, q = 0.5)

    nparticles = length(z0)
    Z = repeat(z0, 1, nsteps+1) # fill with initial condition

    for n = 1:nsteps
        for m = 1:nparticles
            r = rand() 	# random number between 0 and 1
            if r < q
                Z[m, n+1] = Z[m, n] - Δ	# jump left
            else
                Z[m, n+1] = Z[m, n] + Δ	# jump right
            end
        end
    end

    return Z
end

#
Here we give it a quick try with 10 particles and 10,000 steps, using $q = 0.49$ (so it's just slightly biased to the right) and the default step size of $\Delta = 1$.

Each row of the output is the trajectory of a single particle.

In [ ]:
Z = random_walk(z0 = zeros(10), nsteps = 10_000, q = 0.49)

#
If we plot all 10 trajectories at once, we can see the paths that each particle took in this particular run of the simulation.

We see the position of each particle fluctuates randomly, but with a clear tendency to **drift** over time to increasing, positive values.

In [ ]:
series(0:size(Z,2)-1, Z, color = :tab10,
       axis=(xlabel = L"n", ylabel = L"z_n", xlabelsize = 20, ylabelsize = 20))

#
There is another useful way to visualise these results, which is to use a histogram of all particles at different steps $n$.  We'll increase the number of particles to $10,000$ and keep the other parameters unchanged.

By eye, the resulting distributions look to be Gaussian (i.e. Normal), with a mean and variance that are both increasing with $n$.  This is the classic picture for the diffusion of a group of particles all clustered together initially, but spreading apart over time, with nonetheless a net drift to the right.  This could represent, for example, a dump of contaminant particles into a river. The drift is due to the river current and the diffusion is due to random turbulent motion.  Interestingly though, we never input an actual drift term into our code -- instead the drift arose as a consequence of unequal jump probabilities left and right.  Let's keep that in mind.

In [ ]:
Z = random_walk(z0 = zeros(10_000), nsteps = 10_000, q = 0.49)
f = Figure(size=(800,800), fontsize=20)
idxs = [100 1000 ; 2000 10_000] .+ 1
for (pos, n) in pairs(idxs)
  hist(f[Tuple(pos)...], Z[:,n], normalization = :pdf,
       axis=(xlabel = L"z", title=L"n = %$n",), label="histogram")
  xlims!(-600, 600)
end
f

# Fokker-Planck equation

Our numerical simulation suggests that in the appropriate limit, there is a way to describe this random walk process as an evolving Gaussian distribution.  Our statistical intuition also supports this view -- the trajectory of a single particle is determined by the sum of all of its independent, random jumps.  In the limit of many small jumps accumulated over time, the [central limit theorem](https://en.wikipedia.org/wiki/Central_limit_theorem) guarantees that this sum will be Gaussian.

We want to confirm this result analytically.  In doing so, we also aim to deduce precisely how the unequal left/right jump probabilities manifest as a net drift.



Start with a discrete lattice, with spatial step $\Delta$ and discrete time step $\delta t$.  Nothing infinitesimal here.  Let $P(z,t)$ be the probability of a particle being at site $z$ at discrete time $t$.  Everything is discrete, so $P$ is an ordinary probability mass function.  Not a density.

Now consider the probability of a particle being at site $z$ at time $t + \delta t$.  It can only have arrived by jumping from either left or right -- particles cannot remain stationary.  Considering the probability of jumping left is $q$, the one-step **master equation** is

$$
P(z, t + \delta t) = (1-q) P(z - \Delta, t) + q\, P(z + \Delta, t).
$$

In words, a particle can be at position $z$ by having been at position $z - \Delta$ and jumping _right_ (which happens with probability $1-q$), or having been at position $z + \Delta$ and jumping _left_ (which happens with probability $q$).

Thinking ahead to taking the continuum limit, define $u(z,t) = P(z,t) /\Delta$, so that $u(z,t)$ represents a probability _density_ function.  The master equation is linear in $P$, so $u$ satisfies the same equation:

$$
u(z, t + \delta t) = (1-q)\, u(z - \Delta, t) + q\, u(z + \Delta, t).\qquad (*)
$$

Now we Taylor expand.  On the left of the equation, we have
$$
u(z, t + \delta t) = u(z,t) + \delta t \frac{\partial u}{\partial t} + \mathcal{O}(\delta t^2)
$$
and on the right,
$$
u(z \pm \Delta, t) = u(z,t) \pm \Delta \frac{\partial u}{\partial z} + \frac{\Delta^2}{2} \frac{\partial^2 u}{\partial z^2} + \mathcal{O}(\Delta^3)\,.
$$
Substitute both into $(*)$ and simplify:
$$
\delta t \frac{\partial u}{\partial t} = -(1-2q)\Delta \frac{\partial u}{\partial z} + \frac{\Delta^2}{2} \frac{\partial^2 u}{\partial z^2} + \mathcal{O}(\Delta^3 + \delta t^2)
$$
Divide through by $\delta t$:
$$
\frac{\partial u}{\partial t} = -(1-2q)\frac{\Delta}{\delta t} \frac{\partial u}{\partial z} + \frac{\Delta^2}{2 \delta t} \frac{\partial^2 u}{\partial z^2} + \mathcal{O}(\Delta^3/\delta t + \delta t)
$$

Now we will define $v$ and $D$ to be the coefficients in the above formula
$$
v = (1-2q)\frac{\Delta}{\delta t} \qquad\textrm{and}\qquad D = \frac{\Delta^2}{2\delta t}\,.
$$
We recognise $v$ has dimensions of $\textrm{length}/\textrm{time}$ so it is a _velocity_, while $D$ has dimensions of $\textrm{length}^2/\textrm{time}$ so it is a _diffusivity_.  To approach a continuum limit, we need to refine our space step $\Delta \to 0$, time step $\delta t \to 0$ and jump probability $q \to \frac{1}{2}$ so that these parameters remain fixed.  In particular we recognise the diffusive scaling $\Delta^2 \sim \delta t$, which confirms that the remainder term is $\mathcal{O}(\Delta + \delta t)$ and hence goes to zero.

In that limit, we derive one of the most important _partial differential equations_ (PDE) in mathematics, the **Fokker-Planck equation** with constant coefficients:
$$
\frac{\partial u}{\partial t} = -v \frac{\partial u}{\partial z} + D \frac{\partial^2 u}{\partial z^2}\,.
$$
In other contexts (especially fluid mechanics), this equation is referred to as the **advection-diffusion equation**. It describes continuum transport subject to a velocity field (advection, or drift as we called it) and random motion (diffusion). In our application the dependent variable $u$ represents the probability density function of particles, which evolves over time.



## Check our theory

One excellent sanity check on our derivation is to examine the scalings implied by the formulas for the parameters $v$ and $D$.  These are physically measurable quantities -- velocity and diffusion.  So, for example, if we wanted to run the same simulation with `nsteps`$=1000$ instead of `nsteps`$=10,000$, we must do so in a way that keeps $v$ and $D$ unchanged.

This means it is not correct to simply change `nsteps` and be done with it.  Take a look at the two histograms and you can see they're nothing alike!



In [ ]:
# This is the WRONG way to do it, changing only nsteps
Zwrong = random_walk(z0 = zeros(10_000), nsteps = 1000, q = 0.49)

hist(Z[:,end], normalization = :pdf,
     axis=(xlabel = L"z", title="Simulation comparisons done wrong",), label="nsteps = 10,000")
hist!(Zwrong[:,end], normalization = :pdf, label="nsteps = 1000")
axislegend()
current_figure()

## Scaling

The formulas for $v$ and $D$ tell us how to do it properly.
$$
v = (1-2q)\frac{\Delta}{\delta t} \qquad\textrm{and}\qquad D = \frac{\Delta^2}{2\delta t}\,.
$$

Suppose (arbitrarily, since our example is made-up anyway) our final time is $T = 10$.  Then with $n_\textrm{steps}=10,000$ we have
$$
\delta t = \frac{T}{n_\textrm{steps}} = \frac{10}{10,000} = 10^{-3}
$$
and we have also chosen $\Delta = 1$ and $q = 0.49$.  Therefore our effective velocity and diffusivity are
$$
v = (1-2\times 0.49)\frac{1}{10^{-3}} = 20
$$
and
$$
D = \frac{1^2}{2 \times 10^{-3}} = 500\,.
$$

If we now use $n'_\textrm{steps}=1000$ we have
$$
\delta t' = \frac{T}{n'_\textrm{steps}} = \frac{10}{1000} = 10^{-2}
$$
so to keep $v$ and $D$ from changing _we need to also adjust $\Delta$ and $q$ to compensate_!  The new value for $\Delta'$ comes from the formula for $D$:
$$
\Delta' = \sqrt{2D\delta t'} = \sqrt{2 \times 500 \times 10^{-2}} = 3.1623
$$
and the new value for $q$ comes from the formula for $v$:
$$
q' = \frac{1}{2}\left(1 - \frac{v \delta t'}{\Delta'}\right) = \frac{1}{2}\left(1 - \frac{20 \times 10^{-2}}{3.1623}   \right) = 0.4684\,.
$$

The new value of $\Delta'$ is bigger and the new value of $q'$ is further from $\frac{1}{2}$.  This makes sense: with fewer steps in total, the size of each step needs to be larger to compensate, and the bias needs to be more pronounced.

Now the two histograms will overlap, subject only to random variation.

In [ ]:
# This is the right way to do it: change nsteps, q and Δ
Zright = random_walk(z0 = zeros(10_000), nsteps = 1000, q = 0.4684, Δ = 3.1623);

hist(Z[:,end], normalization = :pdf,
     axis=(xlabel = L"z", title="Simulation comparisons done right",), label="nsteps = 10000")
hist!(Zright[:,end], normalization = :pdf, label="nsteps = 1000")
axislegend()
current_figure()

# Analytic solution

Fantastic, we have analysed our biased random walk process and quantified exactly how it leads to a drift (advection) and spread (diffusion) in the particle distribution over time.

Now we would love to overlay the true probability density function $u(z,t)$, the solution to the Fokker-Planck equation
$$
\frac{\partial u}{\partial t} = -v \frac{\partial u}{\partial z} + D \frac{\partial^2 u}{\partial z^2}
$$
over the top of these histograms.  So let's see if we can solve the equation.

If you've studied PDEs you may have learned how to solve this equation using Fourier transforms.  (If so, go ahead!)  But another approach is to use our statistical intuition that the solution must be a Gaussian.  So we propose an **ansatz** (educated guess) that the solution takes the form
$$
u(z,t) = \frac{1}{\sqrt{2\pi s_t^2}}\, \exp \left(- \frac{(z-\mu_t)^2}{2s_t^2}  \right)
$$
i.e. a Gaussian with time-dependent mean $\mu_t$ and time-dependent variance $s_t^2$ (we have to use the symbol $s$, because we have another plan for the symbol $\sigma$ later).

#
Substituting in and solving (exercises!) we find the winning formulas are
$$
\mu_t = z_0 + vt
$$
and
$$
s_t^2 = 2 D t\,.
$$

So the solution is indeed a Gaussian, with mean drifting linearly at rate $v$ and variance increasing linearly at rate $2D$.

We can confirm we have this correct by overlaying this PDF onto our histogram above.

In [ ]:
p_advecdiff(z,t; v,D,z0) = 1/sqrt(4π*D*t) * exp( -(z - z0 - v*t)^2 / (4*D*t) )
lines!(-200..600, z -> p_advecdiff(z, 10; v = 20, D = 500, z0 = 0), color=:black, label="PDF")
axislegend()
current_figure()

# The initial condition
The formula for $u(z,t)$ implies as $t \to 0^+$, the variance approaches zero.  That is, the Gaussian becomes an infinitely tall but infinitesimally thin spike centred at $z = z_0$. Mathematically, $u$ satisfies the **initial condition**
$$
u(z,0)=\delta(z-z_0)
$$

where $\delta(z)$ is called the [Dirac delta function](https://en.wikipedia.org/wiki/Dirac_delta), which we will treat as mathematical shorthand for "infinitely tall, infinitesimally thin spike".

We can overlay the solution at $t = 0.1$ to get a sense for the behaviour as $t \to 0^+$

In [ ]:
lines!(-200..600, z -> p_advecdiff(z, 0.1; v = 20, D = 500, z0 = 0), color=:green, label="PDF (t ≈ 0)")
axislegend()
current_figure()

#
So let's be rigorous about our claim: the solution to the Fokker-Planck equation
$$
\frac{\partial u}{\partial t} = -v \frac{\partial u}{\partial z} + D \frac{\partial^2 u}{\partial z^2}
$$
_subject to the initial condition_
$$
u(z,0)=\delta(z-z_0)
$$
is the Gaussian function
$$
u(z|z_0,t) = \frac{1}{\sqrt{4\pi Dt}}\, \exp \left(- \frac{(z-(z_0 + vt))^2}{4Dt}  \right)\,.
$$

Notice we've switched to using the notation $u(z|z_0,t)$ to really emphasise that this is the PDF for $z$ at time $t$ _given_ that the initial condition was a spike at $z = z_0$.



## What about other initial conditions?

The solution of the PDE subject to some _other_ initial condition $u(z,0) = u_0(z)$ is not in general a Gaussian.  Instead it is the [convolution](https://en.wikipedia.org/wiki/Convolution) of the initial condition with a Gaussian:
$$
u(z,t) = \int_{z_0} u(z|z_0,t)\, u_0(z_0)\, \mathrm{d}z_0\,.
$$
The statistically-minded amongst you will no doubt recognise this as the **law of total probability**: the unconditional distribution is found by integrating over all the conditional ones, weighted by the prior $u_0(z_0)$.

For the applied maths students who haven't done stats for awhile, congratulations, now you can interpret what you knew already about Fourier transforms and convolutions in this probability context.

# Drift done explicitly
It's interesting to observe where the discrete parameters $\Delta$, $\delta t$ and $q$ from our simulation ended up in the continuum solution.  We recall that $v$ and $D$ were defined by holding fixed the quantities
$$
v = (1-2q)\frac{\Delta}{\delta t} \qquad\textrm{and}\qquad D = \frac{\Delta^2}{2\delta t}
$$
in the continuum limit $\Delta \to 0$, $\delta t \to 0$, $q \to 1/2$.  Notably, the diffusivity $D$ has nothing to do with $q$!  The only effect of choosing $q \neq 1/2$ is to add a net drift via the formula for $v$.

You might wonder then, could we arrive at an equivalent description of the process by coding in the drift explicitly in the simulation?  That is, use the discrete update formula
$$
z_{n+1} = \begin{cases}
z_n + d - \Delta,\quad \textrm{with probability } 1/2\\
z_n + d + \Delta,\quad \textrm{with probability } 1/2
\end{cases}
$$
where $d$ is the explicit drift term.  You already know you can! This is exactly how you modelled drift in MXB161 after all.

Here's the code.

In [ ]:
function random_walk_drift(; z0, nsteps, Δ = 1, d = 0)

    nparticles = length(z0)
    Z = repeat(z0, 1, nsteps+1) # fill with initial condition

    for n = 1:nsteps
        for m = 1:nparticles
            r = rand() 	# random number between 0 and 1
            if r < 0.5
                Z[m, n+1] = Z[m, n] + d - Δ	# jump left
            else
                Z[m, n+1] = Z[m, n] + d + Δ	# jump right
            end
        end
    end

    return Z
end

# Sanity check

We calculated earlier that our benchmark simulation is using $v = 20$, so the appropriate value for $d$ satfies
$$
d/ \delta t = v \implies d = v\, \delta t = 20 \times 10^{-3} = 0.02\,.
$$
Let's give it a run to confirm. We can compare against the earlier solution which used unequal $q$.

Sure enough they are doing the same thing in the limit.

In [ ]:
Z2 = random_walk_drift(z0 = zeros(10_000), nsteps = 10_000, Δ = 1, d = 0.02)
hist(Z[:,end], normalization = :pdf,
     axis=(xlabel = L"z", title="Comparison of equivalent discrete models",), label="Model 1")
hist!(Z2[:,end], normalization = :pdf, label="Model 2")
lines!(-200..600, z -> p_advecdiff(z, 10; v = 20, D = 500, z0 = 0), color=:black, label="PDF")
axislegend()
current_figure()

# Equivalent discrete models
This is an important observation: we have derived two different discrete models, which have _the same continuum limit_ described by the same Fokker-Planck equation.

Model 1 (unequal jump probability):
$$
z_{n+1} = \begin{cases}
z_n - \Delta,\quad \textrm{with probability } q\\
z_n + \Delta,\quad \textrm{with probability } 1-q\,.\\
\end{cases}
$$

Model 2 (explicit drift):
$$
z_{n+1} = z_n + d + k_n\Delta,\qquad\textrm{where } P(k_n=\pm 1) = 1/2\,.
$$

We are about to add a third!  Here it is:

Model 3 (normally-distributed jumps):
$$
z_{n+1} = z_n + d + \xi_n,\qquad \textrm{where }\xi_n \sim \mathcal{N}(0,\Delta^2)\,.
$$

That is, Model 3 swaps out Model 2's $\pm \Delta$ and instead uses _normally-distributed_ jumps $\xi_n$ drawn from the distribution $\mathcal{N}(0,\Delta^2)$.  Our claim is that this model produces the same continuum limit as the first two.  And indeed the [central limit theorem](https://en.wikipedia.org/wiki/Central_limit_theorem) assures us this must be so.  In fact, the central limit theorem tells us that _any_ jump distribution with mean $0$ and variance $\Delta^2$ will be equivalent to $\mathcal{N}(0,\Delta^2)$ in the limit of infinitely many steps.

This change to the model actually makes the code a bit tidier, since we no longer need any `if` test to decide between jumping left or right.  The step drawn from the normal distribution will already be negative half the time, and positive half the time.  Here's the code now.

In [ ]:
function random_walk_drift_normal(; z0, nsteps, Δ = 1, d = 0)

    nparticles = length(z0)
    Z = repeat(z0, 1, nsteps+1) # fill with initial condition

    for n = 1:nsteps
        for m = 1:nparticles
            Z[m, n+1] = Z[m, n] + d + Δ * randn()
        end
    end

    return Z
end

#
We'll confirm it does the same thing in the continuum limit as the other two models.

In [ ]:
Z3 = random_walk_drift_normal(z0 = zeros(10_000), nsteps = 10_000, Δ = 1, d = 0.02)
hist!(Z3[:,end], normalization = :pdf, label="Model 3")
axislegend()
current_figure()

#
We now have _three_ discrete models that all give equivalent results in the continuum limit.  Why go to this trouble?  Here's the reason we took this path.  Model 1 was convenient because it kept the particles _on a lattice_.  We had a discrete jump $\Delta$ and we could analyse everything with Taylor expansions around $z \pm \Delta$.  That's how we were able to derive the limiting Fokker-Planck equation for the probability density function.

Model 2 was a stepping stone.  We realised we could model drift with an explicit term, rather than using unequal jump probabilities.  But we lost our lattice restriction at this point. The update formula includes the drift term $d$ and the step $\Delta$, and there's no reason to expect them both to be integers. So the particles aren't confined to a lattice anymore.

With model 3 we are saying, well if we're not on a lattice anyway, let's just pick the step from a normal distribution and be done with it.

But we can still do better!


# Bringing time to the fore

The next improvement we want to make to our model code is to incorporate time $t$ and the continuum parameters $v$ and $D$ explicitly.  We already know the correct relations.  For a step of size $\delta t$,
$$
d = v\, \delta t \qquad\textrm{and}\qquad \Delta^2 = 2 D\, \delta t
$$
so we can just substitute them right in.  Model 3:
$$
z_{n+1} = z_n + d + \xi_n,\qquad \textrm{where }\xi_n \sim \mathcal{N}(0,\Delta^2)
$$
becomes
$$
Z_{t+\delta t} = Z_t + v\, \delta t + \xi_t,\qquad \textrm{where }\xi_t \sim \mathcal{N}(0,2 D\, \delta t)\,.
$$
To ensure we don't get confused you can see we've denoted $Z$ by upper case when it's indexed by time $t$, rather than $z_n$ which is indexed by a count $n$.

Very nice, we have time explicitly in the model, and in place of the scale-dependent parameters $d$ and $\Delta$ we have the nice, fixed, continuum parameters $v$ (velocity) and $D$ (diffusivity).  This isn't quite Model 4 yet, because people usually prefer to draw $\xi_t$ from a _standard_ normal distribution, i.e. mean 0 and variance 1.  If we follow that convention, we would have
$$
Z_{t+\delta t} = Z_t + v\, \delta t + \sqrt{2 D\, \delta t}\, \xi_t,\qquad \textrm{where }\xi_t \sim \mathcal{N}(0,1)\,.
$$
Nearly there!  We can tidy it up a little more by defining a new parameter
$$
\sigma^2 = 2D
$$
and we get our final Model 4.

Model 4 (continuum parameters):
$$
Z_{t+\delta t} = Z_t + v\, \delta t + \sigma\, \sqrt{\delta t}\, \xi_t,\qquad \textrm{where }\xi_t \sim \mathcal{N}(0,1)\,.
$$

Here's the code.

In [ ]:
function random_walk_drift_continuum(; z0, nsteps, T, v, σ)

    nparticles = length(z0)
    Z = repeat(z0, 1, nsteps+1) # fill with initial condition

    δt = T / nsteps  # timestep

    for n = 1:nsteps
        for m = 1:nparticles
            Z[m, n+1] = Z[m, n] + v * δt + σ * sqrt(δt) * randn()
        end
    end

    return Z
end

#
Once again we'll confirm it does the same thing in the continuum limit as the other models.

In [ ]:
Z4 = random_walk_drift_continuum(z0 = zeros(10_000), nsteps=10_000, T = 10, v = 20, σ = sqrt(1000))
hist!(Z4[:,end], normalization = :pdf, label="Model 4")
axislegend()
current_figure()

# Stochastic Differential Equations
Model 4 makes it very convenient to refine the timestep $\delta t$ while keeping everything else consistent. Let's run the simulation with 10 particles for $\delta t \in \{0.1, 0.01, 0.001, 0.0001\}$ and compare the trajectories.

You can see that the general trend of drift and diffusion is comparable across the figures. It's just that the trajectories get "bumpier" as $\delta t$ is reduced.


In [ ]:
f = Figure(size=(800,800), fontsize=20)
nstepsvec = [10  100; 1000 10_000]
for (pos, nsteps) in pairs(nstepsvec)
    Z = random_walk_drift_continuum(; z0 = zeros(10), nsteps, T = 10, v = 20, σ = sqrt(1000))
    series(f[Tuple(pos)...], range(0, 1, length=nsteps+1), Z, color = :tab10,
           axis=(xlabel = L"t", ylabel = L"Z_t", title = L"\delta t = %$(1/nsteps)"))
end
f

#
What would happen to the trajectories as we continued to reduce $\delta t$? In the limit of _infinitesimally small_ $\textrm{d}t$ we could write Model 4 as:
$$
Z_{t+\textrm{d} t} = Z_t + v\, \textrm{d} t + \sigma\, \sqrt{\textrm{d} t}\, \xi_t,\qquad \textrm{where }\xi_t \sim \mathcal{N}(0,1)\,.
$$
Then we would define the **stochastic differentials**
$$
\textrm{d}Z_t := Z_{t + \delta t} - Z_t
$$
and
$$
\textrm{d}W_t := \sqrt{\textrm{d}t}\, \xi_t
$$
and we'd have ourselves a [Stochastic Differential Equation](https://en.wikipedia.org/wiki/Stochastic_differential_equation) (SDE)
$$
\textrm{d}Z_{t} = v\, \textrm{d}t + \sigma\, \textrm{d}W_t\,.
$$

If this is your first ever stochastic differential equation, congratulations!  SDEs are a great new tool to add to your mathematical collection.

Note that it is literally a _stochastic differential_ equation; i.e. $\textrm{d}W_t$ is a **stochastic differential**, with the properties
$$
\begin{align*}
\mathbb{E}[\textrm{d}W_t] &= 0 \\
\mathbb{E}[(\textrm{d}W_t)^2] &= \textrm{d}t \\
\end{align*}
$$
(and more besides, see [Wiener process](https://en.wikipedia.org/wiki/Wiener_process)).

A stochastic differential like $\textrm{d}W_t$ is most definitely _not_ a standard differential like $\textrm{d}t$, and in fact there is a whole new calculus called [Itô calculus](https://en.wikipedia.org/wiki/Ito_calculus) that describes what you can and can't do with stochastic differentials.  We won't be concerned with the technicalities in this unit.

# ODEs vs SDEs
There are several very important differences between Ordinary Differential Equations (ODEs) and Stochastic Differential Equations (SDEs).  For an ODE, something like
$$
\frac{\textrm{d}z}{\textrm{d}t} = f(z,t)
$$

the solution is a _deterministic_ function, $z(t)$.  You plug in a value of $t$, and you get a value $z(t)$.

For an SDE, like our model equation
$$
\textrm{d}Z_{t} = v\, \textrm{d}t + \sigma\, \textrm{d}W_t\,,
$$

the solution is a _stochastic process_, $Z_t$.  You plug in a value of $t$, and you get a _random variable_ $Z_t$.  So really the solution to an SDE is not one function, but a whole _distribution_ of possible trajectories.

And in fact we already know the analytic solution to our SDE, it's
$$
Z_t \sim \mathcal{N}(z_0 + vt, \sigma^2 t)\,.
$$
That's it.  $Z_t$ is a random variable drawn from a normal distribution with mean $z_0 + vt$ and variance $\sigma^2 t$.  That's what counts as the solution to the SDE: you've characterised the random variable $Z_t$ by its distribution. What more could you say!

A second very important difference concerns the notation for SDEs.  It's _so_ tempting to want to "divide through by $\textrm{d}t$" and write the SDE above analogous to an ODE, i.e.
$$
\frac{\textrm{d}Z_t}{\textrm{d}t} = v + \sigma\, \frac{\textrm{d}W_t}{\textrm{d} t}  \qquad \textrm{no no no!}
$$
but _you absolutely cannot do this_.  None of these derivatives exist!

Remember we derived this SDE as the limit of a random walk with smaller and smaller jumps over smaller and smaller timesteps.  The _whole point_ of this limit is that you can zoom in on a trajectory forever and ever and you always see more detail.  This is completely incompatible with taking derivatives.  A sample trajectory of the SDE is in fact differentiable _nowhere_, despite being continuous _everywhere_!

# Conclusion

In this lesson we learned:

* how a biased random walk with small steps leads, in the continuum limit, to an advection-diffusion equation

* how the drift and diffusion coefficients arise from the scalings of the step size, time step, and jump probabilities

* how the Fokker-Planck equation describes the time evolution of a probability density function under drift and noise

* how the same evolution can be described from the particle viewpoint using a stochastic differential equation (SDE)

In the next lesson we will continue our look into diffusion processes, and answer the question: is it possible run a diffusion process in _reverse_?